# GridPulse UK — Silver Carbon-Intensity Transformation

This notebook converts immutable Bronze snapshots into a clean and trusted
Silver dataset.

## Silver rules

1. Reject structurally invalid records.
2. Allow actual carbon intensity to be null for unfinished periods.
3. Group repeated snapshots by electricity period.
4. Prefer a snapshot containing an actual value.
5. Otherwise, use the most recently ingested snapshot.
6. Calculate forecast-error metrics.
7. Maintain one canonical Silver record per electricity period.

In [0]:
BRONZE_TABLE = "gridpulse.bronze.carbon_intensity"
SILVER_TABLE = "gridpulse.silver.carbon_intensity"
REJECTED_TABLE = "gridpulse.silver.carbon_intensity_rejected"

bronze_source_df = spark.table(BRONZE_TABLE)

print(
    f"Bronze records available: "
    f"{bronze_source_df.count()}"
)

bronze_source_df.printSchema()

In [0]:
from pyspark.sql.functions import (
    col,
    lower,
    trim,
    when
)

valid_intensity_indexes = [
    "very low",
    "low",
    "moderate",
    "high",
    "very high"
]

quality_checked_df = (
    bronze_source_df
    .withColumn(
        "rejection_reason",

        when(
            col("record_id").isNull(),
            "missing_record_id"
        )

        .when(
            col("source_system").isNull(),
            "missing_source_system"
        )

        .when(
            col("period_start_utc").isNull(),
            "missing_period_start"
        )

        .when(
            col("period_end_utc").isNull(),
            "missing_period_end"
        )

        .when(
            col("period_end_utc")
            <= col("period_start_utc"),
            "invalid_period_order"
        )

        .when(
            col("forecast_gco2_per_kwh").isNull(),
            "missing_forecast"
        )

        .when(
            col("forecast_gco2_per_kwh") < 0,
            "negative_forecast"
        )

        .when(
            col("actual_gco2_per_kwh") < 0,
            "negative_actual"
        )

        .when(
            ~lower(trim(col("intensity_index")))
            .isin(valid_intensity_indexes),
            "invalid_intensity_index"
        )

        .otherwise(None)
    )
)

In [0]:
valid_bronze_df = (
    quality_checked_df
    .filter(
        col("rejection_reason").isNull()
    )
    .drop("rejection_reason")
)

rejected_bronze_df = (
    quality_checked_df
    .filter(
        col("rejection_reason").isNotNull()
    )
)

valid_count = valid_bronze_df.count()
rejected_count = rejected_bronze_df.count()

print(f"Valid Bronze records: {valid_count}")
print(f"Rejected Bronze records: {rejected_count}")

In [0]:
from pyspark.sql.functions import count

repeated_periods_df = (
    valid_bronze_df
    .groupBy(
        "period_start_utc",
        "period_end_utc"
    )
    .agg(
        count("*").alias("snapshot_count")
    )
    .filter(
        col("snapshot_count") > 1
    )
    .orderBy(
        col("snapshot_count").desc(),
        col("period_start_utc")
    )
)

display(repeated_periods_df)

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import (
    desc,
    row_number
)

period_selection_window = (
    Window
    .partitionBy(
        "period_start_utc",
        "period_end_utc"
    )
    .orderBy(
        desc("actual_available"),
        desc("ingested_at_utc"),
        desc("loaded_at_utc")
    )
)

ranked_snapshots_df = (
    valid_bronze_df
    .withColumn(
        "actual_available",
        when(
            col("actual_gco2_per_kwh").isNotNull(),
            1
        ).otherwise(0)
    )
    .withColumn(
        "selection_rank",
        row_number().over(
            period_selection_window
        )
    )
)

In [0]:
canonical_periods_df = (
    ranked_snapshots_df
    .filter(
        col("selection_rank") == 1
    )
    .drop(
        "actual_available",
        "selection_rank"
    )
)

canonical_count = canonical_periods_df.count()

print(
    f"Canonical electricity periods: "
    f"{canonical_count}"
)

In [0]:
from pyspark.sql.functions import (
    abs as spark_abs,
    concat_ws,
    current_timestamp,
    sha2
)

silver_ready_df = (
    canonical_periods_df
    .select(
        sha2(
            concat_ws(
                "||",
                col("period_start_utc").cast("string"),
                col("period_end_utc").cast("string")
            ),
            256
        ).alias("period_id"),

        col("record_id")
            .alias("source_record_id"),

        col("period_start_utc"),
        col("period_end_utc"),

        col("forecast_gco2_per_kwh"),
        col("actual_gco2_per_kwh"),

        lower(
            trim(col("intensity_index"))
        ).alias("intensity_index"),

        (
            col("actual_gco2_per_kwh")
            - col("forecast_gco2_per_kwh")
        ).alias("forecast_error_gco2_per_kwh"),

        spark_abs(
            col("actual_gco2_per_kwh")
            - col("forecast_gco2_per_kwh")
        ).alias("absolute_forecast_error_gco2_per_kwh"),

        col("ingested_at_utc")
            .alias("selected_ingested_at_utc"),

        current_timestamp()
            .alias("silver_processed_at_utc")
    )
)

display(
    silver_ready_df
    .orderBy("period_start_utc")
    .limit(10)
)

In [0]:
silver_count = silver_ready_df.count()

distinct_period_count = (
    silver_ready_df
    .select("period_id")
    .distinct()
    .count()
)

duplicate_period_ids = (
    silver_count
    - distinct_period_count
)

missing_period_ids = (
    silver_ready_df
    .filter(col("period_id").isNull())
    .count()
)

print(f"Silver rows: {silver_count}")
print(f"Distinct period IDs: {distinct_period_count}")
print(f"Duplicate period IDs: {duplicate_period_ids}")
print(f"Missing period IDs: {missing_period_ids}")

assert silver_count > 0, (
    "Silver contains no records."
)

assert duplicate_period_ids == 0, (
    "Silver contains duplicate period IDs."
)

assert missing_period_ids == 0, (
    "Silver contains missing period IDs."
)

print("Silver in-memory validation passed.")

In [0]:
from delta.tables import DeltaTable

if not spark.catalog.tableExists(SILVER_TABLE):

    (
        silver_ready_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(SILVER_TABLE)
    )

    print(f"Created Silver table: {SILVER_TABLE}")

else:

    silver_delta_table = DeltaTable.forName(
        spark,
        SILVER_TABLE
    )

    (
        silver_delta_table.alias("target")
        .merge(
            silver_ready_df.alias("source"),
            "target.period_id = source.period_id"
        )
        .whenMatchedUpdateAll(
            condition=(
                "target.source_record_id "
                "<> source.source_record_id"
            )
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Updated Silver table: {SILVER_TABLE}")

In [0]:
if not spark.catalog.tableExists(REJECTED_TABLE):

    (
        rejected_bronze_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(REJECTED_TABLE)
    )

    print(
        f"Created rejected-record table: "
        f"{REJECTED_TABLE}"
    )

elif rejected_count > 0:

    rejected_delta_table = DeltaTable.forName(
        spark,
        REJECTED_TABLE
    )

    (
        rejected_delta_table.alias("target")
        .merge(
            rejected_bronze_df.alias("source"),
            "target.record_id = source.record_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(
        f"Merged rejected records into: "
        f"{REJECTED_TABLE}"
    )

else:

    print(
        "No rejected records to merge. "
        "The rejected-record table already exists."
    )

In [0]:
%sql
COMMENT ON TABLE gridpulse.silver.carbon_intensity IS
'Canonical validated national carbon-intensity record for each GB electricity period';

ALTER TABLE gridpulse.silver.carbon_intensity
SET TBLPROPERTIES (
    'quality' = 'silver',
    'source.system' = 'NESO Carbon Intensity API',
    'data.owner' = 'GridPulse',
    'business.key' = 'period_start_utc, period_end_utc'
);

In [0]:
%sql
COMMENT ON TABLE gridpulse.silver.carbon_intensity_rejected IS
'Bronze carbon-intensity records rejected by Silver data-quality rules';

ALTER TABLE gridpulse.silver.carbon_intensity_rejected
SET TBLPROPERTIES (
    'quality' = 'quarantine',
    'source.system' = 'NESO Carbon Intensity API',
    'data.owner' = 'GridPulse'
);

In [0]:
%sql
SELECT
    COUNT(*) AS silver_rows,
    COUNT(DISTINCT period_id) AS distinct_period_ids,
    COUNT(*) - COUNT(DISTINCT period_id) AS duplicate_period_ids
FROM gridpulse.silver.carbon_intensity;

In [0]:
%sql
SELECT
    rejection_reason,
    COUNT(*) AS rejected_records
FROM gridpulse.silver.carbon_intensity_rejected
GROUP BY rejection_reason
ORDER BY rejected_records DESC;

In [0]:
%sql
SELECT
    DATE(period_start_utc) AS electricity_date,
    COUNT(*) AS total_periods,
    COUNT(actual_gco2_per_kwh) AS periods_with_actuals,

    ROUND(
        AVG(forecast_gco2_per_kwh),
        2
    ) AS average_forecast,

    ROUND(
        AVG(actual_gco2_per_kwh),
        2
    ) AS average_actual,

    ROUND(
        AVG(absolute_forecast_error_gco2_per_kwh),
        2
    ) AS mean_absolute_error

FROM gridpulse.silver.carbon_intensity

GROUP BY DATE(period_start_utc)
ORDER BY electricity_date;

In [0]:
%sql
SELECT
    (SELECT COUNT(*)
     FROM gridpulse.bronze.carbon_intensity)
        AS bronze_snapshot_rows,

    (SELECT COUNT(*)
     FROM gridpulse.silver.carbon_intensity)
        AS silver_canonical_rows,

    (SELECT COUNT(*)
     FROM gridpulse.silver.carbon_intensity_rejected)
        AS rejected_rows;

In [0]:
%sql
SELECT COUNT(*) AS silver_rows_after_rerun
FROM gridpulse.silver.carbon_intensity;